In [10]:
from functions import *

In [11]:
a_vec = [763, 679, 397, 61, 697, 373, 
         289, 257, 625, 41, 193, 449]
b_vec = [435, 69, 330, 18, 612, 246, 
         496, 640, 200, 524, 672, 672] 

In [12]:
G = generate_g(a_vec)
all_indices = [i for i in range(l_h**2)]
gb_indices = [3, 8]
ga_indices = [i for i in all_indices if i not in gb_indices]
Ga = G.extract(ga_indices, list(range(G.cols)))
Gb = G.extract(gb_indices, list(range(G.cols)))
basis_a = solve_modular_kernel(Ga, P)
V = Matrix.hstack(*basis_a)

In [13]:
# 1. b_vec を行列形式に変換
b_mat = Matrix(b_vec)

# 2. V * x = b を有理数体（Q）上で解く
# V は main.ipynb で定義された Matrix.hstack(*basis_a)
try:
    # V が正則であれば x = V^-1 * b が求まる
    x_rational = V.solve(b_mat)

    # 3. 有理数の解 a/b を整数 a * inv(b, P) (mod P) に変換する関数
    def to_mod_p(val, p):
        num, den = val.as_numer_denom()
        # Python 3.8+ の pow(den, -1, p) はモジュラ逆数を計算する
        return (int(num) * pow(int(den), -1, p)) % p

    # 各要素に適用
    coefficients = x_rational.applyfunc(lambda v: to_mod_p(v, P))

    print("線形結合の係数ベクトル x:")
    display(coefficients)
    
    # 検算: V * x % P が b_vec と一致するか確認
    check_val = (V * coefficients).applyfunc(lambda x: x % P)
    if check_val == b_mat.applyfunc(lambda x: x % P):
        print("検算成功: 一致しました。")
    else:
        print("警告: 検算に失敗しました。")

except Exception as e:
    print(f"解を求めることができませんでした: {e}")

線形結合の係数ベクトル x:


Matrix([
[  3],
[709],
[689],
[746],
[762],
[732],
[ 68],
[ 84],
[565],
[557],
[744],
[346]])

検算成功: 一致しました。


In [14]:
cycles = generate_cycles(6)
h_x, h_z = generate_h_xz()
constraints = generate_constraints(cycles, a_vec, h_x, h_z)

In [15]:
# 全ての禁止ベクトル（法ベクトル）を個別にリスト化する
unique_forbidden_vectors = []
seen_vectors = set()

# 1. 条件B (潜在部の非可換性) からの制約 r_i
for i in range(Gb.rows):
    c_prime = (Gb.row(i) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T) # 列ベクトルとして保存
        seen_vectors.add(c_tuple)

# 2. 条件C (短いサイクルの回避) からの制約 c_prime
for c in constraints:
    c_prime = (Matrix([c]) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T)
        seen_vectors.add(c_tuple)

print(f"個別に回避すべき禁止制約（超平面）の数: {len(unique_forbidden_vectors)}")

個別に回避すべき禁止制約（超平面）の数: 312


In [16]:
def is_in_general_solution(x_vec, forbidden_vectors, p):
    """
    x_vec がすべての禁止超平面 r_i^T * x = 0 (mod p) を避けているか判定する。
    """
    # ベクトル形式を整える
    x_mat = Matrix(x_vec)
    
    for r in forbidden_vectors:
        # 内積が 0 (mod P) になったらその禁止領域に含まれている
        if (r.T * x_mat)[0] % p == 0:
            return False
    return True

# すでに見つけている特殊解 coefficients (x0) の妥当性を再確認
if is_in_general_solution(coefficients, unique_forbidden_vectors, P):
    print("特殊解 x0 は一般解の条件をすべて満たしています。")

特殊解 x0 は一般解の条件をすべて満たしています。


In [17]:
# --- main.ipynb の続き ---

def get_safe_directions(forbidden_vectors, p):
    """
    禁止領域ベクトル群に対して、なるべく干渉しない（内積が 0 になりにくい）
    基底方向を特定する。
    """
    da = forbidden_vectors[0].rows
    safe_basis = []
    
    # 標準基底 e_j に対して、どれだけの禁止領域と干渉するかスコアリング
    for j in range(da):
        e_j = Matrix.zeros(da, 1)
        e_j[j] = 1
        
        hit_count = 0
        for r in forbidden_vectors:
            if (r.T * e_j)[0] % p == 0:
                hit_count += 1
        
        # 干渉が少ない（hit_count が小さい）基底を優先的に「動かせる方向」とする
        safe_basis.append((hit_count, e_j))
    
    # 干渉の少ない順にソート
    safe_basis.sort(key=lambda x: x[0])
    return [b[1] for b in safe_basis]

# 1. 安全な移動方向のリストを取得
safe_dirs = get_safe_directions(unique_forbidden_vectors, P)

# 2. 一般解の代数的な構成
# x = x0 + sum( alpha_j * safe_dir_j )
# ここで alpha_j を「内積が 0 にならない範囲」に限定する

def construct_smart_solution(x0, safe_dirs, forbidden_vectors, p, max_gen=5):
    solutions = []
    da = x0.rows
    
    # 干渉の少ない上位数個の基底のみを使用して、決定論的に探索
    # ( da=12 すべてを動かすのではなく、安全な数方向に絞る )
    top_safe_dirs = safe_dirs[:4] 
    
    # 小さな係数 (alpha) の組み合わせを試す
    # これは全探索ではなく、安全な軸に沿った「近傍」の探索
    for alphas in product([-1, 0, 1, 2], repeat=len(top_safe_dirs)):
        delta_x = Matrix.zeros(da, 1)
        for alpha, direction in zip(alphas, top_safe_dirs):
            delta_x += alpha * direction
            
        candidate = (x0 + delta_x).applyfunc(lambda v: v % p)
        
        if is_in_general_solution(candidate, forbidden_vectors, p):
            # 重複チェックをして追加
            if not any(candidate == s for s in solutions):
                solutions.append(candidate)
                if len(solutions) >= max_gen:
                    break
    return solutions

# スマートに新しい解を生成
smart_solutions = construct_smart_solution(coefficients, safe_dirs, unique_forbidden_vectors, P)

print(f"代数的な方向選択により、{len(smart_solutions)} 個の解が特定されました。")
if smart_solutions:
    print("代表的な一般解の係数 (x):")
    display(smart_solutions[0])

代数的な方向選択により、2 個の解が特定されました。
代表的な一般解の係数 (x):


Matrix([
[  3],
[709],
[689],
[746],
[762],
[732],
[ 68],
[ 84],
[565],
[557],
[744],
[346]])

In [18]:
# 生成された一般解の一つを b_vec に戻して確認
if smart_solutions:
    for sample_x in smart_solutions:
        # b = V * x (mod P)
        b_vec = (V * sample_x).applyfunc(lambda val: val % P)
        
        print("b_vec:", list(b_vec))
        print("check:", check(a_vec, b_vec), '\n')



b_vec: [435, 69, 330, 18, 612, 246, 496, 640, 200, 524, 672, 672]
check: True 

b_vec: [429, 747, 726, 78, 540, 618, 16, 128, 56, 564, 96, 352]
check: True 

